<a href="https://colab.research.google.com/github/hanauert/Hausarbeit-GenAI/blob/main/06_stance_detection_binary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Load gold annotated dataset**

In [1]:
import pandas as pd
from transformers import pipeline

In [2]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
subsample_gold = pd.read_csv('/content/drive/MyDrive/FKM/stance_detection/gold_annotated.csv')

In [4]:
adj_to_base = {
    1:'nicht negativ',
    2:'negativ',
    3:'nicht negativ',
}

# Apply mapping
subsample_gold['gold_standard_binary'] = subsample_gold['gold_standard'].map(adj_to_base)

In [5]:
subsample_gold['lead'] = subsample_gold['body_text'].str.split('\n\n|\n').str[0]

In [6]:
subsample_gold.head()

,metadata,body_text,title,date,month,newspaper,article_id,paragraph,gold_standard,gold_standard_binary,lead
0,Wir verschenken zu viel Zeit\n \nFrankfurter R...,Hochwasser in Niedersachsen im Januar 2024. dp...,Wir verschenken zu viel Zeit,2025-01-02,2025-01,FR,A_1870,Das gilt übrigens auch für die demokratischen ...,1.0,nicht negativ,Hochwasser in Niedersachsen im Januar 2024. dpa
1,Das Armutsrisiko in Deutschland ist rückläufig...,Im Wahlkampf nimmt die Debatte um Arm und Reic...,Das Armutsrisiko in Deutschland ist rückläufig...,2024-12-26,2024-12,Welt,A_291,Trotz des gewachsenen Migrantenanteils in der ...,3.0,nicht negativ,Im Wahlkampf nimmt die Debatte um Arm und Reic...
2,Arbeitgeber erwarten Wohlstandsverlust durch P...,Bahnverbindungen und Unterrichtsstunden fallen...,Arbeitgeber erwarten Wohlstandsverlust durch P...,2023-11-26,2023-11,Welt,B_22,Die Unionsfraktion hatte stattdessen vorgeschl...,2.0,negativ,Bahnverbindungen und Unterrichtsstunden fallen...
3,Das Ende einer Erfolgsstory; Kanzler Scholz fe...,"Es ist einer der Textbausteine, die Bundeskanz...",Das Ende einer Erfolgsstory; Kanzler Scholz fe...,2024-11-25,2024-11,Welt,A_303,Hauptgründe für die Aufwärtsbewegung sind die ...,1.0,nicht negativ,"Es ist einer der Textbausteine, die Bundeskanz..."
4,„Investitionen entscheiden über die Leistungsf...,Die neue Landesregierung ist mit dem Versprech...,„Investitionen entscheiden über die Leistungsf...,2024-09-10,2024-09,FR,B_408,Die Aufnahme einer Erwerbsarbeit ist die beste...,1.0,nicht negativ,Die neue Landesregierung ist mit dem Versprech...


##**Define Metrics**

In [7]:
from sklearn.metrics import cohen_kappa_score, f1_score, accuracy_score, precision_score, recall_score
import pandas as pd

def evaluate_model(true, pred):
    return {
        'Cohens Kappa': cohen_kappa_score(true, pred),
        'F1': f1_score(true, pred, average='weighted'),
        'Accuracy': accuracy_score(true, pred),
        'Precision': precision_score(true, pred, average='weighted'),
        'Recall': recall_score(true, pred, average='weighted')
    }

#**Binary Classification**

#**Binary: mlburnham/Political_DEBATE_DeBERTa_large_v1.1**

In [8]:
classifier_mlb = pipeline("zero-shot-classification", model='mlburnham/Political_DEBATE_DeBERTa_large_v1.1')

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

Device set to use cuda:0


###**Annotate Title + Paragraph**

In [9]:
samples = list("Title:\n" + subsample_gold['title'] + "\n\n" + subsample_gold['paragraph']) #+ subsample_gold['lead'] + "\n\n"
template = 'Migrantinnen und Migranten werden in dem Text eher {} dargestellt.'
labels = ['als Problem', 'nicht als Problem']

In [10]:
# classify the documents
res = classifier_mlb(samples, labels, hypothesis_template = template, multi_label = False)

# return the most probable label and add it to our data frame
subsample_gold['label_mlb_p8_bin'] = [label['labels'][0] for label in res]

subsample_gold['scores_mlb_p8_bin'] = [r['scores'][0] for r in res]

In [11]:
adj_to_base = {
    'nicht als Problem':'nicht negativ',
    'als Problem':'negativ',
}

# Apply mapping
subsample_gold['label_mlb_p8_bin'] = subsample_gold['label_mlb_p8_bin'].map(adj_to_base)

In [12]:
from sklearn.metrics import classification_report
print(classification_report(subsample_gold['gold_standard_binary'], subsample_gold['label_mlb_p8_bin']))

               precision    recall  f1-score   support

      negativ       0.64      0.71      0.67        62
nicht negativ       0.90      0.87      0.88       188

     accuracy                           0.83       250
    macro avg       0.77      0.79      0.78       250
 weighted avg       0.84      0.83      0.83       250



In [15]:
metrics_mlb = evaluate_model(subsample_gold['gold_standard_binary'], subsample_gold['label_mlb_p8_bin'])

print("Evaluation Metrics: mlburnham/Political_DEBATE_DeBERTa_large_v1.1 Binary")
for metric, value in metrics_mlb.items():
    print(f"{metric}: {value:.3f}")

Evaluation Metrics: mlburnham/Political_DEBATE_DeBERTa_large_v1.1 Binary
Cohens Kappa: 0.556
F1: 0.831
Accuracy: 0.828
Precision: 0.835
Recall: 0.828


#**Binary: Gemma3:4b**


In [16]:
!curl https://ollama.ai/install.sh | sh

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13281    0 13281    0     0  63922      0 --:--:-- --:--:-- --:--:-- 64159
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [17]:
import subprocess

# Start Ollama service in the background
ollama_process = subprocess.Popen(["ollama", "serve"])

# Optional: check if it's running
print("Ollama server started in the background.")

Ollama server started in the background.


In [18]:
!curl http://localhost:11434

Ollama is running

In [19]:
!pip install -q ollama openai
import pandas as pd
import ollama
from ollama import chat

In [20]:
ollama.pull("gemma3:4b")

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [21]:
from ollama import chat
from pydantic import BaseModel
from typing import List, Literal
import json

class MigrationAnnotation(BaseModel):
   annotation: Literal['negativ', 'nicht negativ']


def classify_with_gemma_structured_binary(text):
    messages = [
    {
        "role": "system",
        "content": (
            "Stance Detection: Klassifiziere den folgenden Absatz.\n"
            "negativ: Migration wird im Text eher kritisch betrachtet.\n"
            "nicht negativ: Migration wird in dem Text positiv oder neutral betrachtet.\n"
        )
    },
    {
        "role": "user",
        "content": f"{text} --- Wie ist die Haltung des Autors zu Migration? "
    }
]

    response = chat(
        messages=messages,
        model="gemma3:4b",
        format=MigrationAnnotation.model_json_schema(),
        options=dict(seed=42, temperature=0.0)
    )

    return response['message']['content']

###**Annotate Title + Paragraph**

In [22]:
# Apply the classification function the DataFrame
subsample_gold['gemma_bin'] = (subsample_gold['title'] + "\n\n" + subsample_gold['paragraph']).apply(classify_with_gemma_structured_binary)

subsample_gold['gemma_bin'] = subsample_gold['gemma_bin'].str.extract(r'"annotation"\s*:\s*"([^"]+)"')

In [23]:
subsample_gold.head()

,metadata,body_text,title,date,month,newspaper,article_id,paragraph,gold_standard,gold_standard_binary,lead,label_mlb_p8_bin,scores_mlb_p8_bin,gemma_bin
0,Wir verschenken zu viel Zeit\n \nFrankfurter R...,Hochwasser in Niedersachsen im Januar 2024. dp...,Wir verschenken zu viel Zeit,2025-01-02,2025-01,FR,A_1870,Das gilt übrigens auch für die demokratischen ...,1.0,nicht negativ,Hochwasser in Niedersachsen im Januar 2024. dpa,nicht negativ,0.999986,negativ
1,Das Armutsrisiko in Deutschland ist rückläufig...,Im Wahlkampf nimmt die Debatte um Arm und Reic...,Das Armutsrisiko in Deutschland ist rückläufig...,2024-12-26,2024-12,Welt,A_291,Trotz des gewachsenen Migrantenanteils in der ...,3.0,nicht negativ,Im Wahlkampf nimmt die Debatte um Arm und Reic...,nicht negativ,0.999999,nicht negativ
2,Arbeitgeber erwarten Wohlstandsverlust durch P...,Bahnverbindungen und Unterrichtsstunden fallen...,Arbeitgeber erwarten Wohlstandsverlust durch P...,2023-11-26,2023-11,Welt,B_22,Die Unionsfraktion hatte stattdessen vorgeschl...,2.0,negativ,Bahnverbindungen und Unterrichtsstunden fallen...,negativ,0.828434,negativ
3,Das Ende einer Erfolgsstory; Kanzler Scholz fe...,"Es ist einer der Textbausteine, die Bundeskanz...",Das Ende einer Erfolgsstory; Kanzler Scholz fe...,2024-11-25,2024-11,Welt,A_303,Hauptgründe für die Aufwärtsbewegung sind die ...,1.0,nicht negativ,"Es ist einer der Textbausteine, die Bundeskanz...",nicht negativ,0.956105,negativ
4,„Investitionen entscheiden über die Leistungsf...,Die neue Landesregierung ist mit dem Versprech...,„Investitionen entscheiden über die Leistungsf...,2024-09-10,2024-09,FR,B_408,Die Aufnahme einer Erwerbsarbeit ist die beste...,1.0,nicht negativ,Die neue Landesregierung ist mit dem Versprech...,nicht negativ,0.551716,negativ


In [24]:
from sklearn.metrics import classification_report
print(classification_report(subsample_gold['gold_standard_binary'], subsample_gold['gemma_bin']))

               precision    recall  f1-score   support

      negativ       0.39      0.76      0.51        62
nicht negativ       0.88      0.61      0.72       188

     accuracy                           0.64       250
    macro avg       0.64      0.68      0.62       250
 weighted avg       0.76      0.64      0.67       250



#**Evaluation Binary**

In [25]:
metrics_gemma = evaluate_model(subsample_gold['gold_standard_binary'], subsample_gold['gemma_bin'])

print("Evaluation Metrics: Gemma3:4b Binary")
for metric, value in metrics_gemma.items():
    print(f"{metric}: {value:.3f}")

Evaluation Metrics: Gemma3:4b Binary
Cohens Kappa: 0.276
F1: 0.668
Accuracy: 0.644
Precision: 0.761
Recall: 0.644


In [26]:
metrics_df = pd.DataFrame([metrics_gemma, metrics_mlb],
                          index=['gemma3:4b_binary', 'mlburnham_binary',])

print(metrics_df.round(3))

                  Cohens Kappa     F1  Accuracy  Precision  Recall
gemma3:4b_binary         0.276  0.668     0.644      0.761   0.644
mlburnham_binary         0.556  0.831     0.828      0.835   0.828


#**Annotate Full Dataset**

In [27]:
paragraphs_merged_df = pd.read_csv('/content/drive/MyDrive/FKM/stance_detection/df_newspaper_filtered_by_paragraph_mergedAB.csv')

In [28]:
paragraphs_merged_df['lead'] = paragraphs_merged_df['body_text'].str.split('\n\n|\n').str[0]

In [29]:
paragraphs_merged_df.head()

,metadata,body_text,title,date,month,newspaper,article_id,paragraph,lead,labels_NLI,scores_NLI
0,- Rembrandts Amsterdam. Goldene Zeiten? - Drib...,Im 17. Jahrhundert ist Amsterdam die Metropole...,- Rembrandts Amsterdam. Goldene Zeiten? - Drib...,2024-12-03,2024-12,FR,A_1,Die Ausstellung hinterfragt die traditionelle ...,Im 17. Jahrhundert ist Amsterdam die Metropole...,nicht negativ,0.999842
1,300 MODELLE IN 15 FARBEN - Wachsender Markt fü...,90 Prozent der Perücken sind aus „Premium-Synt...,300 MODELLE IN 15 FARBEN - Wachsender Markt fü...,2024-09-06,2024-09,FR,A_10,Die Büro- und Sozialräume im Gebäude sind mode...,90 Prozent der Perücken sind aus „Premium-Synt...,nicht negativ,0.999995
2,Abhängige von der Straße holen\n \nFrankfurter...,"VON STEVEN MICKSCH\nRäume zum Drogenkonsum, ni...",Abhängige von der Straße holen,2024-11-20,2024-11,FR,A_20,Noch eine Etage weiter oben soll es dann Notsc...,VON STEVEN MICKSCH,nicht negativ,0.993135
3,Alles German\n \nFrankfurter Rundschau\nDienst...,"In Hahndorf in Südaustralien, rund 30 Kilomete...",Alles German,2025-02-10,2025-02,FR,A_39,Im 19. Jahrhundert und bis weit ins 20. Jahrhu...,"In Hahndorf in Südaustralien, rund 30 Kilomete...",nicht negativ,0.999999
4,Alles für die Staatsräson\n \nFrankfurter Rund...,Die Berliner Ausländerbehörde bereitet die Abs...,Alles für die Staatsräson,2025-04-07,2025-04,FR,A_40,Vertreter:innen des Anwaltsteams sehen Paralle...,Die Berliner Ausländerbehörde bereitet die Abs...,nicht negativ,0.983100


##**Load mlburnham/Political_DEBATE_DeBERTa_large_v1.1 if not already loaded**

In [30]:
classifier_mlb = pipeline("zero-shot-classification", model='mlburnham/Political_DEBATE_DeBERTa_large_v1.1', batch_size = 8)

Device set to use cuda:0


In [31]:
samples = list("Title:\n" + paragraphs_merged_df['title'] + "\n\n" + paragraphs_merged_df['paragraph'])
template = 'Migrantinnen und Migranten werden in dem Text eher {} dargestellt.'
labels = ['als Problem', 'nicht als Problem']

In [32]:
# classify the documents
res = classifier_mlb(samples, labels, hypothesis_template = template, multi_label = False)

# return the most probable label and add it to the data frame
paragraphs_merged_df['labels_NLI'] = [label['labels'][0] for label in res]

paragraphs_merged_df['scores_NLI'] = [r['scores'][0] for r in res]

In [33]:
adj_to_base = {
    'nicht als Problem':'nicht negativ',
    'als Problem':'negativ',
}

# Apply mapping
paragraphs_merged_df['labels_NLI'] = paragraphs_merged_df['labels_NLI'].map(adj_to_base)

In [34]:
paragraphs_merged_df['labels_NLI'].value_counts()

,count
labels_NLI,
nicht negativ,589
negativ,279


In [35]:
paragraphs_merged_df.head()

,metadata,body_text,title,date,month,newspaper,article_id,paragraph,lead,labels_NLI,scores_NLI
0,- Rembrandts Amsterdam. Goldene Zeiten? - Drib...,Im 17. Jahrhundert ist Amsterdam die Metropole...,- Rembrandts Amsterdam. Goldene Zeiten? - Drib...,2024-12-03,2024-12,FR,A_1,Die Ausstellung hinterfragt die traditionelle ...,Im 17. Jahrhundert ist Amsterdam die Metropole...,nicht negativ,0.999842
1,300 MODELLE IN 15 FARBEN - Wachsender Markt fü...,90 Prozent der Perücken sind aus „Premium-Synt...,300 MODELLE IN 15 FARBEN - Wachsender Markt fü...,2024-09-06,2024-09,FR,A_10,Die Büro- und Sozialräume im Gebäude sind mode...,90 Prozent der Perücken sind aus „Premium-Synt...,nicht negativ,0.999995
2,Abhängige von der Straße holen\n \nFrankfurter...,"VON STEVEN MICKSCH\nRäume zum Drogenkonsum, ni...",Abhängige von der Straße holen,2024-11-20,2024-11,FR,A_20,Noch eine Etage weiter oben soll es dann Notsc...,VON STEVEN MICKSCH,nicht negativ,0.993135
3,Alles German\n \nFrankfurter Rundschau\nDienst...,"In Hahndorf in Südaustralien, rund 30 Kilomete...",Alles German,2025-02-10,2025-02,FR,A_39,Im 19. Jahrhundert und bis weit ins 20. Jahrhu...,"In Hahndorf in Südaustralien, rund 30 Kilomete...",nicht negativ,0.999999
4,Alles für die Staatsräson\n \nFrankfurter Rund...,Die Berliner Ausländerbehörde bereitet die Abs...,Alles für die Staatsräson,2025-04-07,2025-04,FR,A_40,Vertreter:innen des Anwaltsteams sehen Paralle...,Die Berliner Ausländerbehörde bereitet die Abs...,nicht negativ,0.983100


##**save labels**

In [36]:
paragraphs_merged_df.to_csv('/content/drive/MyDrive/FKM/stance_detection/df_newspaper_filtered_by_paragraph_mergedAB.csv', index=False)